# `financialtools.processor` — User Guide

Core data-acquisition and metric-evaluation module: fetches raw financials from Yahoo Finance,
reshapes them into long-format DataFrames, computes 24 scored metrics plus 14 unscored
extended metrics, scores the 24 on a 1–5 scale, and detects red flags.

```
financialtools/processor.py
│
├── RateLimiter(per_minute, per_hour, per_day)
│   └── acquire()               block until a call slot is available
│
├── Downloader(ticker)
│   ├── from_ticker(ticker)     classmethod — fetch + reshape; never raises
│   ├── get_merged_data()       long-format DataFrame (balance sheet + income + cashflow)
│   ├── get_info_data()         DataFrame with marketCap, forwardPE, etc.
│   ├── combine_merged_data()   classmethod — concat across multiple Downloader instances
│   ├── combine_info_data()     classmethod — concat info DataFrames
│   └── stream_download()       classmethod — yield Downloaders one by one, save to Parquet
│
├── FundamentalTraderAssistant(data, weights)
│   ├── evaluate()              full pipeline → dict with 6 keys
│   ├── compute_metrics()       compute 24 SCORED_METRICS columns
│   ├── compute_extended_metrics()  14 unscored metrics (efficiency, growth, red-flag ratios)
│   ├── compute_valuation_metrics()  P/E, P/B, P/FCF, EarningsYield, FCFYield
│   ├── compute_scores()        score each metric 1–5
│   ├── raw_red_flags()         cash-flow quality flags
│   └── metrics_red_flags()     threshold-based flags
│
└── Module-level helpers
    ├── SCORED_METRICS          list[str] — 24 metric column names
    ├── _EMPTY_RESULT_KEYS      tuple[str] — canonical evaluate() output keys (6)
    └── _empty_result()         factory → {k: pd.DataFrame() for k in _EMPTY_RESULT_KEYS}
```

| Symbol | Needs yfinance | Needs LLM | Best for |
|---|---|---|---|
| `RateLimiter` | no | no | throttling any external API call loop |
| `Downloader.from_ticker` | **yes** | no | fetching one ticker's financials |
| `Downloader.get_merged_data` | no | no | accessing already-fetched data |
| `FundamentalTraderAssistant` | no | no | scoring metrics from any merged DataFrame |
| `_empty_result` | no | no | safe fallback / test scaffolding |

**Prerequisites:**
- Sections 1–3 and 7–9 run with **no API key** and **no internet connection**.
- Sections 4–6 require `yfinance` and an active internet connection.

## Sections
1. [Setup](#1-setup)
2. [Module-level constants](#2-module-level-constants)
3. [RateLimiter](#3-ratelimiter)
4. [FundamentalTraderAssistant — construction](#4-fundamentaltraderassistant--construction)
5. [FundamentalTraderAssistant — evaluate()](#5-fundamentaltraderassistant--evaluate)
6. [Downloader (Live)](#6-downloader-live)
7. [Error handling](#7-error-handling)
8. [Common failure modes](#8-common-failure-modes)
9. [End-to-end pattern](#9-end-to-end-pattern)

## 1 — Setup

In [17]:
import logging
import numpy as np
import pandas as pd

from financialtools.processor import (
    RateLimiter,
    Downloader,
    FundamentalTraderAssistant,
    SCORED_METRICS,
    _EMPTY_RESULT_KEYS,
    _empty_result,
)
from financialtools.exceptions import EvaluationError

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)
print("Imports OK")

Imports OK


## 2 — Module-level constants

`SCORED_METRICS` is a reference list of the **24** metric column names produced by
`compute_metrics()` (original 11 + 13 extended scored metrics). It is kept for
documentation and test-scaffolding purposes.
`evaluate()` and `compute_scores()` **derive `value_vars` dynamically** from
`compute_metrics()` output columns (all columns that are not `ticker`, `time`, or `sector`),
so adding a new metric to `compute_metrics()` automatically includes it in scoring without
editing `SCORED_METRICS`.

`_EMPTY_RESULT_KEYS` is the tuple of **6** keys that every `evaluate()` return value must
contain. `_empty_result()` is the factory that creates a fresh dict of empty DataFrames
with exactly those keys.

> Always call `_empty_result()` — never copy `_EMPTY_RESULT` directly — because a shallow
> copy would share DataFrame objects across calls.

In [18]:
print(f"SCORED_METRICS ({len(SCORED_METRICS)} items):")
for m in SCORED_METRICS:
    print(f"  {m}")

print(f"\n_EMPTY_RESULT_KEYS : {_EMPTY_RESULT_KEYS}")

er = _empty_result()
print(f"\n_empty_result() keys  : {list(er.keys())}")
print(f"All values empty      : {all(v.empty for v in er.values())}")
print(f"Distinct objects      : {len({id(v) for v in er.values()})} (should equal {len(er)})")

SCORED_METRICS (11 items):
  GrossMargin
  OperatingMargin
  NetProfitMargin
  EBITDAMargin
  ROA
  ROE
  FCFToRevenue
  FCFYield
  FCFtoDebt
  DebtToEquity
  CurrentRatio

_EMPTY_RESULT_KEYS : ('metrics', 'eval_metrics', 'composite_scores', 'raw_red_flags', 'red_flags')

_empty_result() keys  : ['metrics', 'eval_metrics', 'composite_scores', 'raw_red_flags', 'red_flags']
All values empty      : True
Distinct objects      : 5 (should equal 5)


## 3 — RateLimiter

`RateLimiter` is a thread-safe sliding-window rate limiter. It tracks call timestamps and
sleeps the calling thread until all three windows (minute, hour, day) have a free slot.

### Constructor

```python
RateLimiter(per_minute=60, per_hour=360, per_day=8000)
```

| Parameter | Default | Meaning |
|---|---|---|
| `per_minute` | 60 | max calls within any 60-second window |
| `per_hour` | 360 | max calls within any 3600-second window |
| `per_day` | 8000 | max calls within any 86400-second window |

### acquire()

Blocks until all three windows have a free slot, then records the timestamp. The internal
`calls` list is pruned to the last 24 hours on every `acquire()` call to bound memory usage.

### Thread safety

All state mutations (reading + writing `self.calls`) are protected by `threading.Lock`.
It is safe to share one `RateLimiter` instance across threads — for example across a
`ThreadPoolExecutor` that downloads multiple tickers in parallel.

### Typical usage

Pass a `RateLimiter` instance to `Downloader.stream_download()` to pace bulk downloads.

In [19]:
import time
import threading

# --- Basic construction ---
limiter = RateLimiter(per_minute=10, per_hour=100, per_day=1000)
print(f"per_minute : {limiter.per_minute}")
print(f"per_hour   : {limiter.per_hour}")
print(f"per_day    : {limiter.per_day}")
print(f"calls so far: {len(limiter.calls)}")

# --- acquire() under the limit — should return immediately ---
for i in range(3):
    limiter.acquire()
    print(f"  call {i+1}: acquired at {time.strftime('%H:%M:%S')}")

print(f"calls recorded: {len(limiter.calls)}")

per_minute : 10
per_hour   : 100
per_day    : 1000
calls so far: 0
  call 1: acquired at 22:24:58
  call 2: acquired at 22:24:58
  call 3: acquired at 22:24:58
calls recorded: 3


In [20]:
# --- Thread-safety demo: 5 threads sharing one RateLimiter with per_minute=5 ---
# Each thread calls acquire() once. With per_minute=5, the first 5 calls go through
# immediately; a 6th would block. Here we fire exactly 5 to show safe concurrent use.

results = []
lock = threading.Lock()

def worker(tid):
    limiter_shared.acquire()
    with lock:
        results.append(f"thread-{tid} acquired at {time.strftime('%H:%M:%S')}")

limiter_shared = RateLimiter(per_minute=5, per_hour=100, per_day=1000)
threads = [threading.Thread(target=worker, args=(i,)) for i in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

for r in sorted(results):
    print(r)
print(f"Total calls recorded: {len(limiter_shared.calls)}")

thread-0 acquired at 22:24:58
thread-1 acquired at 22:24:58
thread-2 acquired at 22:24:58
thread-3 acquired at 22:24:58
thread-4 acquired at 22:24:58
Total calls recorded: 5


## 4 — FundamentalTraderAssistant — construction

`FundamentalTraderAssistant` validates its inputs eagerly in `__init__` and raises
`EvaluationError` for any of three guard conditions:

| Guard | Raises when |
|---|---|
| Empty ticker | `data['ticker'].dropna().unique()` is empty |
| Multiple tickers | `data` contains more than one distinct non-null ticker value |
| Empty sector in weights | `weights['sector'].dropna().unique()` is empty |
| Multiple sectors in weights | `weights` contains more than one distinct sector |

All three attributes start as empty DataFrames:

```python
self.metrics       = pd.DataFrame()
self.eval_metrics  = pd.DataFrame()
self.scores        = pd.DataFrame()
```

They are populated lazily by `compute_metrics()`, `compute_valuation_metrics()`, and
`evaluate()`.

### Live data via Downloader

The cells below download real AAPL financials using `Downloader.from_ticker()`.
`get_merged_data()` returns balance-sheet + income-statement + cash-flow joined on
`(ticker, time)`, **and automatically broadcasts market-data columns** (`marketcap`,
`currentprice`, `sharesoutstanding`) from `_info` across all time periods.
No manual `get_info_data()` merge is required before constructing `FundamentalTraderAssistant`.
Sector weights come from `config.sec_sector_metric_weights`.

> **Requires internet.** `from_ticker()` never raises — on failure it returns an empty
> `Downloader` and logs the error, so always check whether the result DataFrames are empty.

In [21]:
# Download AAPL financials — requires yfinance and an active internet connection.
# from_ticker() never raises; on failure it returns an empty Downloader and logs the error.
TICKER = "AAPL"
SECTOR = "technology"   # key in sec_sector_metric_weights (lowercase)

d      = Downloader.from_ticker(TICKER)
merged = d.get_merged_data()
info   = d.get_info_data()

print(f"merged shape : {merged.shape}")   # (time_periods, financial_columns)
print(f"info   shape : {info.shape}")     # (1, ~100 yfinance info fields)

if merged.empty:
    raise RuntimeError(f"Download failed for {TICKER} — check logs/error.log")

merged[["ticker", "time", "total_revenue", "gross_profit", "free_cash_flow"]].head()

merged shape : (5, 166)
info   shape : (1, 184)


,ticker,time,total_revenue,gross_profit,free_cash_flow
0,AAPL,2021-09-30,NaN,NaN,NaN
1,AAPL,2022-09-30,3.943280e+11,1.707820e+11,1.114430e+11
2,AAPL,2023-09-30,3.832850e+11,1.691480e+11,9.958400e+10
3,AAPL,2024-09-30,3.910350e+11,1.806830e+11,1.088070e+11
4,AAPL,2025-09-30,4.161610e+11,1.952010e+11,9.876700e+10


In [ ]:
# get_merged_data() automatically merges marketcap, currentprice, and sharesoutstanding
# from _info across all time periods — no manual enrichment needed.
print(f"data shape : {merged.shape}")

market_cols = [c for c in Downloader._MARKET_COLS if c in merged.columns]
missing     = [c for c in Downloader._MARKET_COLS if c not in merged.columns]
if missing:
    print(f"Warning: market columns not found in merged data: {missing}")

print(f"Market columns present: {market_cols}")
merged[["ticker", "time"] + market_cols].head()

In [ ]:
from financialtools.config import sec_sector_metric_weights

# Build the weights DataFrame from the canonical sector config — single source of truth.
# AAPL is in "technology" per sec_sector_metric_weights (yfinance sectorKey convention).
sector_weights_dict = sec_sector_metric_weights[SECTOR]
weights = pd.DataFrame({
    "sector" : SECTOR,
    "metrics": list(sector_weights_dict.keys()),
    "weights": list(sector_weights_dict.values()),
})

print(f"weights shape : {weights.shape}")

# Construct the assistant — merged already includes marketcap, currentprice,
# sharesoutstanding from get_merged_data(); no separate info merge required.
fta = FundamentalTraderAssistant(data=merged, weights=weights)
print(f"\nfta.ticker  : {fta.ticker}")
print(f"fta.sector  : {fta.sector}")
print(f"fta.metrics (before evaluate) empty : {fta.metrics.empty}")
print(f"fta.scores  (before evaluate) empty : {fta.scores.empty}")

## 5 — FundamentalTraderAssistant — evaluate()

`evaluate()` runs the full scoring pipeline and returns a dict with **six** keys:

| Key | Type | Content |
|---|---|---|
| `metrics` | `pd.DataFrame` | wide — one row per `(ticker, time)`, 24 scored metric columns |
| `eval_metrics` | `pd.DataFrame` | wide — valuation metrics: P/E, P/B, P/FCF, EarningsYield, FCFYield |
| `composite_scores` | `pd.DataFrame` | one row per `(sector, ticker, time)` with `composite_score` |
| `raw_red_flags` | `pd.DataFrame` | cash-flow quality flags (negative FCF/OCF, EBITDA >> OCF) |
| `red_flags` | `pd.DataFrame` | threshold-based flags (negative margins, high D/E, etc.) |
| `extended_metrics` | `pd.DataFrame` | 14 unscored columns: efficiency chain, growth rates, red-flag ratios |

### Invariant: never raises

`evaluate()` wraps its entire body in `try/except`. On any internal failure it logs the
error and returns `_empty_result()` — a dict of **six** empty DataFrames — so the caller
always receives the same shape.

### Composite score formula

```
composite_score = sum(score_i * weight_i) / sum(weight_i)
```

where `score_i` is the 1–5 score for metric `i` (from `score_metric()`) and `weight_i`
comes from the `weights` DataFrame passed to the constructor.

### Scored vs unscored split

**24 scored metrics** feed the composite score. **14 unscored extended metrics**
(returned under `extended_metrics`) do not — they are time-differential (`pct_change`) or
derived chains (e.g. CCC) that lack universal thresholds. Four scored metrics use
**inverse scoring** (lower value → higher score): `DebtToEquity`, `DebtRatio`,
`NetDebtToEBITDA`, `CapexRatio`.

In [ ]:
result = fta.evaluate()

print("Result keys:", list(result.keys()))
print(f"All keys present: {set(result.keys()) == set(_EMPTY_RESULT_KEYS)}")

print(f"\n--- metrics ({result['metrics'].shape}) ---")
print(result["metrics"][["ticker", "time", "GrossMargin", "ROE", "DebtToEquity"]].to_string(index=False))

print(f"\n--- eval_metrics ({result['eval_metrics'].shape}) ---")
print(result["eval_metrics"][["ticker", "time", "P/E", "P/B", "FCFYield"]].to_string(index=False))

print(f"\n--- composite_scores ({result['composite_scores'].shape}) ---")
print(result["composite_scores"].to_string(index=False))

print(f"\n--- extended_metrics ({result['extended_metrics'].shape}) ---")
ext = result["extended_metrics"]
print(ext[["ticker", "time", "DSO", "DIO", "DPO", "CCC", "RevenueGrowth"]].to_string(index=False))

In [25]:
# Inspect red flags from real AAPL data.
raw_rf = result["raw_red_flags"]
rf     = result["red_flags"]

print(f"raw_red_flags rows : {len(raw_rf)}")
if not raw_rf.empty:
    print(raw_rf.to_string(index=False))
else:
    print("  (none — all cash flows positive)")

print(f"\nred_flags rows     : {len(rf)}")
if not rf.empty:
    print(rf.to_string(index=False))
else:
    print("  (none — all metrics within thresholds)")

# Force a red-flag scenario by injecting bad values into the first two rows.
# This exercises the detection logic independently of what the live data contains.
data_bad = data.copy()
data_bad.loc[data_bad.index[0], "free_cash_flow"] = -500_000_000
data_bad.loc[data_bad.index[1], "net_income_common_stockholders"] = -200_000_000

fta_bad    = FundamentalTraderAssistant(data=data_bad, weights=weights)
result_bad = fta_bad.evaluate()

print("\n--- raw_red_flags with injected negative FCF ---")
print(result_bad["raw_red_flags"].to_string(index=False))

print("\n--- red_flags with injected negative net margin ---")
print(result_bad["red_flags"].to_string(index=False))

raw_red_flags rows : 0
  (none — all cash flows positive)

red_flags rows     : 1
ticker       time      metrics                 red_flag
  AAPL 2022-09-30 DebtToEquity High Debt-to-Equity (>2)

--- raw_red_flags with injected negative FCF ---
ticker       time metrics                red_flag
  AAPL 2021-09-30 rrf_fcf Negative Free Cash Flow

--- red_flags with injected negative net margin ---
ticker       time         metrics                 red_flag
  AAPL 2022-09-30 NetProfitMargin      Negative Net Margin
  AAPL 2022-09-30             ROA             Negative ROA
  AAPL 2022-09-30             ROE             Negative ROE
  AAPL 2022-09-30    DebtToEquity High Debt-to-Equity (>2)
